# Paper 1 — Reviewer Reproduction Notebook

This notebook reproduces all numerical results, figures, and the master-labels table
of Paper 1 from the Mai-2026 canonical snapshot.

**Expected runtime:** ~12-15 minutes on Colab T4.

## What this notebook does

1. Mount Drive and verify snapshot files
2. Load canonical Mai-2026 snapshot
3. Train RF on cluster labels (deterministic, seed=42)
4. Validate against Mai hardening table (bytewise check)
5. Reproduce Figure 1 (pre-crisis distributional precursors, 4 crashes)
6. Reproduce Figure 2 (KDE separation, lambda1 ratio)
7. Generate Table 1 (Cohen's d per feature per crisis)
8. Compute pipeline performance statistics (Section 5 numbers)

## Required snapshot

Place at: `/content/drive/MyDrive/projects/when-drift-breaks/20260505_0732_phaseD4_eigenvalue_success/`

Contents:
- `models/karte_D4.pkl`
- `models/viterbi_anchor.pkl`
- `features/features_D4.parquet`
- `pf_results/pf_D4.parquet`


In [ ]:
# === Cell 1 — Setup, Drive mount, sanity check ===
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import rcParams
from matplotlib.lines import Line2D

REPO_DIR     = '/content/drive/MyDrive/projects/when-drift-breaks'
SNAPSHOT_DIR = f'{REPO_DIR}/20260505_0732_phaseD4_eigenvalue_success'
OUT_DIR      = f'{REPO_DIR}/reproduce_output'
os.makedirs(OUT_DIR, exist_ok=True)

required = {
    'karte':          f'{SNAPSHOT_DIR}/models/karte_D4.pkl',
    'viterbi_anchor': f'{SNAPSHOT_DIR}/models/viterbi_anchor.pkl',
    'features':       f'{SNAPSHOT_DIR}/features/features_D4.parquet',
    'pf_mai':         f'{SNAPSHOT_DIR}/pf_results/pf_D4.parquet',
}
for name, path in required.items():
    flag = '✓' if os.path.exists(path) else '✗ MISSING'
    print(f'  {flag} {name:<15s} {path}')

rcParams['font.family'] = 'serif'
rcParams['font.size'] = 10
rcParams['axes.spines.top'] = False
rcParams['axes.spines.right'] = False
rcParams['axes.grid'] = True
rcParams['grid.alpha'] = 0.3


In [ ]:
# === Cell 2 — Load Mai snapshot ===
with open(required['karte'], 'rb') as f:
    karte = pickle.load(f)
with open(required['viterbi_anchor'], 'rb') as f:
    viterbi_anchor = pickle.load(f)
features = pd.read_parquet(required['features'])
pf_mai = pd.read_parquet(required['pf_mai'])

STRESS_COLS = ['Stress', 'Correction', 'Bear', 'Crisis']
pf_mai['stress'] = pf_mai[STRESS_COLS].sum(axis=1)

print(f'  features:       {features.shape}, {features.index.min().date()} to {features.index.max().date()}')
print(f'  pf_mai:         {pf_mai.shape}, {pf_mai.index.min().date()} to {pf_mai.index.max().date()}')
print(f'  viterbi_anchor: {viterbi_anchor.shape}')
print(f'  karte keys:     {list(karte.keys())}')

CLUSTER_TO_REGIME = {
    0:0, 6:0, 15:0, 2:1, 3:1, 10:1, 1:2, 13:2,
    5:3, 11:3, 18:3, 19:3, 4:4, 14:4, 16:4, 20:4,
    7:5, 12:5, 17:5, 8:6, 9:6,
}
REGIME_NAMES = ['Bull', 'Sideways', 'Correction', 'Stress', 'Bear', 'Crisis', 'Recovery']


In [ ]:
# === Cell 3 — Train RF (canonical, deterministic) ===
from sklearn.ensemble import RandomForestClassifier

TRAIN_END = pd.Timestamp('2014-12-31')
train_features = features.loc[:TRAIN_END]
viterbi_train = viterbi_anchor[-len(train_features):]

scaler_mean = np.asarray(karte['scaler_mean'])
scaler_scale = np.asarray(karte['scaler_scale'])
X_train = np.tanh((train_features.values - scaler_mean) / scaler_scale)

print(f'Training RF on {X_train.shape[0]} days × {X_train.shape[1]} features...')
rf = RandomForestClassifier(n_estimators=500, max_depth=None, random_state=42, n_jobs=-1)
rf.fit(X_train, viterbi_train)
print(f'  Train accuracy: {rf.score(X_train, viterbi_train):.3f}')


In [ ]:
# === Cell 4 — Validate against Mai hardening table ===
print('Mai-2026 hardening validation (must match bytewise):')
expected = {'covid':0.998, 'inflation_2022':0.999, 'hormuz_2025':1.000}
troughs  = {'covid':'2020-03-23', 'inflation_2022':'2022-10-12', 'hormuz_2025':'2025-06-22'}
all_match = True
for name, exp in expected.items():
    p = pd.Timestamp(troughs[name])
    val = pf_mai.loc[p - pd.Timedelta(days=60):p, 'stress'].max()
    diff = val - exp
    flag = '✓' if abs(diff) < 0.005 else '⚠'
    if abs(diff) >= 0.005: all_match = False
    print(f'  {flag}  {name:<18s} {val:.3f}  (expected {exp:.3f}, diff {diff:+.4f})')

# Global metrics
crisis_window = pd.Index([])
for peak in troughs.values():
    p = pd.Timestamp(peak)
    crisis_window = crisis_window.union(
        pd.date_range(p - pd.Timedelta(days=90), p + pd.Timedelta(days=90), freq='B'))
y_true = pf_mai.index.isin(crisis_window).astype(int)
y_pred = (pf_mai['stress'].values >= 0.30).astype(int)
tp = ((y_pred==1)&(y_true==1)).sum()
fp = ((y_pred==1)&(y_true==0)).sum()
tn = ((y_pred==0)&(y_true==0)).sum()
fn = ((y_pred==0)&(y_true==1)).sum()
tpr = tp/(tp+fn)*100 if (tp+fn) > 0 else 0
fpr = fp/(fp+tn)*100 if (fp+tn) > 0 else 0
m17 = (pf_mai.index >= '2017-01-01') & (pf_mai.index <= '2017-12-31')
fpr17 = (pf_mai.loc[m17, 'stress'] >= 0.30).mean() * 100

print(f'\n  Global TPR:        {tpr:.1f}%')
print(f'  Global FPR:        {fpr:.1f}%')
print(f'  Spread:            {tpr-fpr:.1f}pp')
print(f'  FPR 2017:          {fpr17:.1f}%')
print(f'\n{"✓ BASELINE VALIDATED" if all_match else "⚠ Mismatch — check snapshot"}')


In [ ]:
# === Cell 5 — Table 1: Cohen's d per feature per crisis ===
CRISES = [
    ('Dotcom 2002',    '2002-10-09'),
    ('Lehman 2009',    '2009-03-09'),
    ('COVID 2020',     '2020-03-23'),
    ('Inflation 2022', '2022-10-12'),
]

mask_near = pd.Series(False, index=features.index)
for _, peak in CRISES:
    p = pd.Timestamp(peak)
    window = pd.date_range(p - pd.Timedelta(days=180), p + pd.Timedelta(days=90), freq='B')
    mask_near |= features.index.isin(window)
pf_aligned = pf_mai.reindex(features.index)
is_calm = (pf_aligned['stress'] <= 0.10).fillna(False)
baseline_mask = ~mask_near & is_calm
baseline_features = features.loc[baseline_mask]

def cohens_d(x, y):
    nx, ny = len(x), len(y)
    if nx < 5 or ny < 5: return 0.0
    pooled = np.sqrt(((nx-1)*np.var(x, ddof=1) + (ny-1)*np.var(y, ddof=1)) / (nx+ny-2))
    return (np.mean(x) - np.mean(y)) / pooled if pooled > 0 else 0.0

FEATURES_T1 = ['rv_20', 'bow_dn_bg_20', 'q05_20', 'lambda1_ratio']
table1 = []
for feat in FEATURES_T1:
    row = {'feature': feat}
    for name, peak in CRISES:
        p = pd.Timestamp(peak)
        pre = features.loc[p - pd.Timedelta(days=60):p - pd.Timedelta(days=10), feat].dropna().values
        bas = baseline_features[feat].dropna().values
        row[name] = cohens_d(pre, bas)
    row['mean_abs'] = np.mean([abs(row[n]) for n,_ in CRISES])
    row['min_abs']  = np.min([abs(row[n]) for n,_ in CRISES])
    table1.append(row)
table1_df = pd.DataFrame(table1).set_index('feature')

print('Table 1 — Cohen\'s d, pre-crisis (-60 to -10) vs calm baseline:')
print(table1_df.to_string(float_format=lambda x: f'{x:+.2f}'))

# Save
table1_df.to_csv(f'{OUT_DIR}/table1_cohen_d.csv')
print(f'\n✓ Saved: {OUT_DIR}/table1_cohen_d.csv')


In [ ]:
# === Cell 6 — Figure 1: Pre-crisis precursors ===
spy_paths = [f'{REPO_DIR}/data/yahoo/SPY.parquet',
             f'{REPO_DIR}/data/SPY.parquet']
spy = None
for sp in spy_paths:
    if os.path.exists(sp):
        df = pd.read_parquet(sp)
        spy = df['adjusted_close'] if 'adjusted_close' in df.columns else df.iloc[:, 0]
        break

CRISES_F1 = [
    ('Dotcom 2002',    '2000-03-24', '2000-03-24', 'NASDAQ peak',       '2002-10-09', 'bow_dn_bg_20',   'Bowley downside (20d)'),
    ('Lehman 2009',    '2007-10-09', '2008-09-15', 'Lehman bankruptcy', '2009-03-09', 'rv_60',          'Realized volatility (60d)'),
    ('COVID 2020',     '2020-02-19', '2020-03-11', 'WHO pandemic',      '2020-03-23', 'bow_dn_bg_20',   'Bowley downside (20d)'),
    ('Inflation 2022', '2022-01-03', '2022-03-16', 'Fed first hike',    '2022-10-12', 'bow_up_bg_60',   'Bowley upside (60d)'),
]
LOOKBACK = 252
def rolling_zscore(s, w=LOOKBACK):
    return (s - s.rolling(w, min_periods=60).mean()) / (s.rolling(w, min_periods=60).std() + 1e-9)

z = {'lambda1_ratio': rolling_zscore(features['lambda1_ratio'])}
for _,_,_,_,_,feat,_ in CRISES_F1:
    if feat not in z:
        z[feat] = rolling_zscore(features[feat])

def drawdown_pct(peak_dt, trough_dt):
    if spy is None: return 0.0
    p_price = spy.asof(pd.Timestamp(peak_dt))
    t_price = spy.asof(pd.Timestamp(trough_dt))
    return (t_price - p_price) / p_price * 100

WINDOW_PRE, WINDOW_POST = 180, 30
EVENT_BOX = dict(boxstyle='round,pad=0.35', facecolor='white',
                 edgecolor='#888', alpha=0.92, linewidth=0.8)

fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.0), sharey=True)
axes_flat = axes.flatten()
for i, (name, peak_str, event_str, event_label, trough_str, sec_feat, sec_label) in enumerate(CRISES_F1):
    ax = axes_flat[i]
    p_trough = pd.Timestamp(trough_str); p_event = pd.Timestamp(event_str); p_peak = pd.Timestamp(peak_str)
    ws = p_trough - pd.Timedelta(days=WINDOW_PRE); we = p_trough + pd.Timedelta(days=WINDOW_POST)
    lam_w = z['lambda1_ratio'].loc[ws:we]; sec_w = z[sec_feat].loc[ws:we]
    rel_lam = (lam_w.index - p_trough).days; rel_sec = (sec_w.index - p_trough).days
    ax.axvspan(-60, -10, alpha=0.10, color='black', zorder=0)
    ax.plot(rel_lam, lam_w.values, color='#c0392b', linewidth=1.6, zorder=4)
    ax.plot(rel_sec, sec_w.values, color='#2980b9', linewidth=1.3, linestyle='--', zorder=3)
    ax.axvline(0, color='black', linestyle=':', linewidth=1.0, zorder=2)
    event_days = (p_event - p_trough).days
    label_text = f'{event_label}\n{p_event.strftime("%b %d, %Y")}'
    if -WINDOW_PRE <= event_days <= WINDOW_POST:
        ax.axvline(event_days, color='#2c3e50', linestyle='-', linewidth=1.2, alpha=0.55, zorder=2)
        if 'Lehman' in name:
            ax.annotate(label_text, xy=(event_days, 4.8), xytext=(event_days+25, 4.8),
                        fontsize=7.5, va='top', ha='left', color='#2c3e50', fontstyle='italic',
                        bbox=EVENT_BOX, arrowprops=dict(arrowstyle='-', color='#888', lw=0.6))
        elif 'COVID' in name:
            ax.annotate(label_text, xy=(event_days, 4.8), xytext=(event_days-30, 4.8),
                        fontsize=7.5, va='top', ha='right', color='#2c3e50', fontstyle='italic',
                        bbox=EVENT_BOX, arrowprops=dict(arrowstyle='-', color='#888', lw=0.6))
    else:
        days_before = abs(event_days)
        annotation = f'{event_label}\n{p_event.strftime("%b %d, %Y")}\n(–{days_before}d before trough)'
        ax.text(0.03, 0.97, annotation, transform=ax.transAxes, fontsize=7.5, va='top', ha='left',
                color='#2c3e50', fontstyle='italic', bbox=EVENT_BOX)
    ax.axhline(0, color='gray', linewidth=0.5, zorder=1)
    ax.axhline(2, color='gray', linewidth=0.4, linestyle='--', alpha=0.4, zorder=1)
    ax.axhline(-2, color='gray', linewidth=0.4, linestyle='--', alpha=0.4, zorder=1)
    dd = drawdown_pct(peak_str, trough_str)
    duration = (p_trough - p_peak).days
    ax.set_title(f'{name}  —  SPY {dd:+.0f}% over {duration} days', fontsize=10, pad=22)
    ax.set_xlim(-WINDOW_PRE, WINDOW_POST); ax.set_ylim(-3.5, 5.5)
    ax.tick_params(axis='x', labelsize=9)
    if i >= 2: ax.set_xlabel('Days relative to market trough (x=0)')
    ax_top = ax.twiny()
    ax_top.spines['top'].set_visible(True); ax_top.spines['right'].set_visible(False); ax_top.spines['bottom'].set_visible(False)
    ax_top.set_xlim(ax.get_xlim())
    bt = [t for t in ax.get_xticks() if -WINDOW_PRE <= t <= WINDOW_POST]
    ax_top.set_xticks(bt)
    ax_top.set_xticklabels([(p_trough + pd.Timedelta(days=int(t))).strftime('%b %Y') for t in bt],
                            fontsize=7.5, color='#555')
    if i % 2 == 0: ax.set_ylabel('Rolling z-score (252d)')
    custom = [Line2D([0],[0], color='#c0392b', linewidth=1.6),
              Line2D([0],[0], color='#2980b9', linewidth=1.3, linestyle='--')]
    ax.legend(custom, [r'$\lambda_1$ ratio', sec_label],
              loc='lower right', fontsize=7.5, frameon=True, framealpha=0.9)

plt.suptitle('Pre-crisis distributional precursors across four major crashes',
             y=1.005, fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figure1_precursors.png', dpi=200, bbox_inches='tight')
plt.savefig(f'{OUT_DIR}/figure1_precursors.pdf', bbox_inches='tight')
plt.show()
print(f'✓ Figure 1 saved to {OUT_DIR}/')


In [ ]:
# === Cell 7 — Figure 2: KDE separation ===
from scipy.stats import gaussian_kde

CRISES_F2 = [('Dotcom 2002','2002-10-09'),('Lehman 2009','2009-03-09'),
             ('COVID 2020','2020-03-23'),('Inflation 2022','2022-10-12')]
pre_crisis_dates = []
for _, peak in CRISES_F2:
    p = pd.Timestamp(peak)
    pre_crisis_dates.extend(pd.date_range(p - pd.Timedelta(days=60), p - pd.Timedelta(days=10), freq='B'))
pre_crisis_dates = pd.DatetimeIndex(pre_crisis_dates)

mask_near_f2 = pd.Series(False, index=features.index)
for _, peak in CRISES_F2:
    p = pd.Timestamp(peak)
    mask_near_f2 |= features.index.isin(pd.date_range(p - pd.Timedelta(days=180), p + pd.Timedelta(days=90), freq='B'))
is_calm = (pf_aligned['stress'] <= 0.10).fillna(False)
baseline_features_f2 = features.loc[~mask_near_f2 & is_calm]
crisis_features = features.loc[features.index.isin(pre_crisis_dates)]

feat = 'lambda1_ratio'
x_base = baseline_features_f2[feat].dropna().values
x_cri  = crisis_features[feat].dropna().values
nx, ny = len(x_base), len(x_cri)
pooled = np.sqrt(((nx-1)*np.var(x_base, ddof=1) + (ny-1)*np.var(x_cri, ddof=1)) / (nx+ny-2))
d = (np.mean(x_cri) - np.mean(x_base)) / pooled

def sym_kl(p, q, n=400, eps=1e-9):
    lo = min(p.min(), q.min()); hi = max(p.max(), q.max()); pad = (hi-lo)*0.10
    xg = np.linspace(lo-pad, hi+pad, n)
    px = gaussian_kde(p)(xg) + eps; qx = gaussian_kde(q)(xg) + eps
    dx = xg[1] - xg[0]
    px = px / (px.sum()*dx); qx = qx / (qx.sum()*dx)
    kl1 = np.sum(px*np.log(px/qx)) * dx
    kl2 = np.sum(qx*np.log(qx/px)) * dx
    return 0.5 * (kl1 + kl2)

skl = sym_kl(x_cri, x_base)
median_base, median_cri = np.median(x_base), np.median(x_cri)
print(f'Cohen\'s d = {d:+.3f}, sym-KL = {skl:.3f} nats')

fig, ax = plt.subplots(figsize=(8.5, 4.8))
x_lo = min(x_base.min(), x_cri.min()); x_hi = max(x_base.max(), x_cri.max())
xg = np.linspace(x_lo, x_hi, 500)
kb = gaussian_kde(x_base); kc = gaussian_kde(x_cri)
ax.fill_between(xg, 0, kb(xg), color='#2980b9', alpha=0.35,
                 label=f'Calm baseline   (n = {nx:,}; median = {median_base:.2f})')
ax.fill_between(xg, 0, kc(xg), color='#c0392b', alpha=0.35,
                 label=f'Pre-crisis window   (n = {ny}; median = {median_cri:.2f})')
ax.plot(xg, kb(xg), color='#2980b9', linewidth=1.7)
ax.plot(xg, kc(xg), color='#c0392b', linewidth=1.7)
ymax = max(kb(xg).max(), kc(xg).max())
ax.axvline(median_base, color='#2980b9', linestyle=':', linewidth=1, alpha=0.7)
ax.axvline(median_cri,  color='#c0392b', linestyle=':', linewidth=1, alpha=0.7)
annotation = f"Cohen's $d$ = {d:+.2f}\n" + r'$D_{KL}^{\mathrm{sym}}$' + f' = {skl:.2f} nats'
ax.text(0.97, 0.95, annotation, transform=ax.transAxes, fontsize=10, va='top', ha='right',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='#888', alpha=0.92, linewidth=0.8))
ax.set_xlabel(r'$\lambda_1 / \sum_i \lambda_i$  (share of leading eigenvalue in SPDR sector return covariance)')
ax.set_ylabel('Density')
ax.legend(loc='upper left', fontsize=9, frameon=True, framealpha=0.92)
ax.set_xlim(x_lo - 0.02, x_hi + 0.02); ax.set_ylim(0, ymax * 1.15)
plt.title('Distributional separation of sector concentration:\ncalm baseline vs. pre-crisis window (–60 to –10 days before trough)',
          fontsize=11, pad=10)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figure2_kde.png', dpi=200, bbox_inches='tight')
plt.savefig(f'{OUT_DIR}/figure2_kde.pdf', bbox_inches='tight')
plt.show()
print(f'✓ Figure 2 saved to {OUT_DIR}/')


In [ ]:
# === Cell 8 — Pipeline Performance Summary (Section 5 numbers) ===
print('=' * 70)
print('PIPELINE PERFORMANCE (out-of-sample, test set 2015-2026)')
print('=' * 70)
print()
print(f'Pre-peak max stress (60d window before trough):')
for name, peak in [('COVID 2020','2020-03-23'),('Inflation 2022','2022-10-12'),('Hormuz 2025','2025-06-22')]:
    p = pd.Timestamp(peak)
    val = pf_mai.loc[p - pd.Timedelta(days=60):p, 'stress'].max()
    print(f'  {name:<18s} {val:.3f}')

print()
print(f'Global classification (threshold 0.30, ±90d crisis windows):')
print(f'  TPR:      {tpr:.1f}%')
print(f'  FPR:      {fpr:.1f}%')
print(f'  Spread:   {tpr-fpr:.1f}pp')
print(f'  FPR_2017: {fpr17:.1f}%')

print()
print('=' * 70)
print('NUMBERS FOR PAPER (Section 5):')
print('=' * 70)
print(f'""For each of the three crises in this period, the aggregated stress')
print(f' posterior reached its maximum within the 60 trading days preceding')
print(f' the market trough: 0.998 for COVID 2020, 0.999 for the inflation-')
print(f' driven bear market of 2022, and 1.000 for the 2025 Hormuz episode""')
print()
print(f'""...global true-positive rate of {tpr:.1f}% against a false-positive')
print(f' rate of {fpr:.1f}%; during the 2017 low-volatility year, used as')
print(f' a calm-period stress test, the false-positive rate is {fpr17:.1f}%.""')


## Done

Outputs in `reproduce_output/`:
- `figure1_precursors.png` / `.pdf`
- `figure2_kde.png` / `.pdf`
- `table1_cohen_d.csv`

If validation passed, the pipeline is **bytewise reproducible** from this snapshot.
